# Assignment 1 — QANet

**COMP5329 / COMP4329 Deep Learning — University of Sydney, Semester 1 2026**

Run the notebook from top to bottom. The data download and preprocessing sections automatically skip themselves if the required files already exist in the repository folder.

This notebook is the submission entry point for training and evaluation.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = "/content/drive/MyDrive/Assignment1_2026"
except Exception:
    PROJECT_ROOT = "."

print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
# Install Python dependencies (run once per session)
!pip install -r {PROJECT_ROOT}/requirements.txt -q
!python -m spacy download en_core_web_sm

---
## Section 0 — Environment Setup

Mount Google Drive, install dependencies, and configure the working directory.

In [ ]:
import os
import sys
import torch

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

---
## Section 1 — Download Data

The notebook downloads the mini SQuAD + GloVe package only if the raw data files are missing.

In [ ]:
from pathlib import Path
from Tools.download import download_mini

raw_files = [
    Path("_data/squad/train-mini.json"),
    Path("_data/squad/dev-v1.1.json"),
    Path("_data/glove/glove.mini.txt"),
]

if all(path.exists() for path in raw_files):
    print("Raw data files already exist; skipping download.")
else:
    download_mini(data_dir="_data")

---
## Section 2 — Preprocess Data

The notebook preprocesses the dataset only if the expected cached files are missing.

In [ ]:
from pathlib import Path
from Tools.preproc import preprocess

processed_files = [
    Path("_data/train.npz"),
    Path("_data/dev.npz"),
    Path("_data/word_emb.json"),
    Path("_data/char_emb.json"),
    Path("_data/train_eval.json"),
    Path("_data/dev_eval.json"),
]

if all(path.exists() for path in processed_files):
    print("Preprocessed files already exist; skipping preprocessing.")
else:
    preprocess(
        train_file="_data/squad/train-mini.json",
        dev_file="_data/squad/dev-v1.1.json",
        glove_word_file="_data/glove/glove.mini.txt",
        target_dir="_data",
        para_limit=400,
        ques_limit=50,
    )

---
## Section 3 — Train

Trains the corrected QANet model and saves the best checkpoint to `_model/model.pt`.

In [ ]:
import shutil
from TrainTools.train import train

for path in ("_model", "_log"):
    if os.path.exists(path):
        shutil.rmtree(path)

results = train(
    train_npz="_data/train.npz",
    dev_npz="_data/dev.npz",
    word_emb_json="_data/word_emb.json",
    char_emb_json="_data/char_emb.json",
    train_eval_json="_data/train_eval.json",
    dev_eval_json="_data/dev_eval.json",
    save_dir="_model",
    log_dir="_log",
    num_steps=3000,
    checkpoint=500,
    val_num_batches=25,
    test_num_batches=25,
    batch_size=8,
    seed=42,
    early_stop=20,
    optimizer_name="adam",
    scheduler_name="lambda",
    loss_name="qa_nll",
    norm_name="layer_norm",
    norm_groups=8,
    activation="relu",
    init_name="kaiming",
)

print(f"Best F1: {results['best_f1']:.4f}  |  Best EM: {results['best_em']:.4f}")

---
## Section 4 — Evaluate

Loads the saved checkpoint and evaluates it on the full dev set.

In [ ]:
from EvaluateTools.evaluate import evaluate

metrics = evaluate(
    dev_npz="_data/dev.npz",
    word_emb_json="_data/word_emb.json",
    char_emb_json="_data/char_emb.json",
    dev_eval_json="_data/dev_eval.json",
    save_dir="_model",
    log_dir="_log",
    ckpt_name="model.pt",
    batch_size=8,
    test_num_batches=-1,
    loss_name="qa_nll",
    norm_name="layer_norm",
    norm_groups=8,
    activation="relu",
    init_name="kaiming",
)

print(f"F1: {metrics['f1']:.4f}  |  EM: {metrics['exact_match']:.4f}  |  Loss: {metrics['loss']:.6f}")